In [ ]:
import ee
import pandas as pd

ee.Authenticate()
ee.Initialize(project='ee-sample1234')
point = ee.Geometry.Point([74.166375, 31.079083])  # lon, lat of your field

def collection_split(d1,d2,band):
  collection = (ee.ImageCollection("NASA/SMAP/SPL3SMP_E/005")
                .select(band)
                .filterDate(d1,d2))
  return collection



def get_data(start, end, band):
    collection = collection_split(start, end, band)

    def extract(img):
      val = img.reduceRegion(
          reducer=ee.Reducer.mean(),
          geometry=point,
          scale=9000
      ).get(band)
      return ee.Feature(None, {
          'date': img.date().format('YYYY-MM-dd'),
          'sm': val
      })
    features = collection.map(extract)
    data = features.getInfo()['features']


    df = pd.DataFrame([f['properties'] for f in data])

    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date')
    print(df)
    return df



am_data = get_data('2018-01-01','2023-01-01', 'soil_moisture_am')

           date        sm
0    2018-01-01       NaN
1    2018-01-02  0.268640
2    2018-01-03       NaN
3    2018-01-04       NaN
4    2018-01-05  0.186763
...         ...       ...
1740 2022-12-27       NaN
1741 2022-12-28       NaN
1742 2022-12-29  0.255321
1743 2022-12-30       NaN
1744 2022-12-31       NaN

[1745 rows x 2 columns]


In [ ]:
pm_data = get_data('2018-01-01','2023-01-01', 'soil_moisture_pm')

           date        sm
0    2018-01-01       NaN
1    2018-01-02       NaN
2    2018-01-03  0.145370
3    2018-01-04       NaN
4    2018-01-05       NaN
...         ...       ...
1740 2022-12-27  0.257940
1741 2022-12-28       NaN
1742 2022-12-29       NaN
1743 2022-12-30  0.189812
1744 2022-12-31       NaN

[1745 rows x 2 columns]


In [ ]:
df_merged = pd.merge(am_data, pm_data, on='date', how='outer')  # sm_x = df1, sm_y = df2

In [ ]:
df_merged['sm'] = df_merged[['sm_x', 'sm_y']].mean(axis=1, skipna=True)

In [ ]:
merged_sm = df_merged['sm']
am_sm = am_data['sm']
pm_sm = pm_data['sm']

In [ ]:
final_sm_list = [m for m in merged_sm if m==m]
final_am_list = [m for m in am_sm if m==m]
final_pm_list = [m for m in pm_sm if m==m]

In [ ]:
import numpy as np

low_per = 5
up_per = 95

print(np.percentile(final_sm_list, low_per))
print(np.percentile(final_sm_list, up_per))
print("")
print(np.percentile(final_am_list, low_per))
print(np.percentile(final_am_list, up_per))
print("")
print(np.percentile(final_pm_list, low_per))
print(np.percentile(final_pm_list, up_per))

0.1539871022105217
0.3597012460231781

0.16084986925125122
0.3745051622390747

0.14526173025369643
0.34797442704439163
